# Notebook de evaluación — ADAG1 · Proyecto Oráculo · G01

**Integrantes:** Andrés Santiago Restrepo Duque · Keyla Daniela Cartagena Díaz

Carga la configuración de `entrega.json` y la evalúa con el oráculo. **No hay que editar código, rutas ni parámetros.**

## Datos de prueba

El notebook busca, en este orden, el primer archivo que exista:

1. la ruta indicada en la variable de entorno `ORACULO_DATOS`;
2. `/content/datos_prueba.json`;
3. `/content/drive/MyDrive/datos_prueba.json`.

Para evaluar con datos privados basta con **subir el archivo como `datos_prueba.json` al panel de archivos de Colab** (`/content`) antes de ejecutar la sección 3. Debe tener el mismo formato de instancia que `datos_visibles.json` (`id`, `prompt`, `ids`, `kwargs` y, opcional, `restriccion`).

Si no encuentra ninguno, evalúa sobre la partición de **validación** de `datos_visibles.json` (87 instancias) y lo avisa en la salida.

## Cómo ejecutarlo

1. Colab ▸ *Entorno de ejecución* ▸ *Cambiar tipo de entorno* ▸ **T4 GPU**.
2. Ejecutar la sección 1 y **reiniciar la sesión**.
3. Ejecutar el resto de celdas en orden.

## 1 · Instalación  ·  *al terminar, reiniciar la sesión*

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!if [ ! -d /content/open-instruct ]; then git clone -q https://github.com/allenai/open-instruct /content/open-instruct; fi
!if [ -d /content/ADA-2026-2-GP01_Andres_y_Keyla ]; then git -C /content/ADA-2026-2-GP01_Andres_y_Keyla pull -q; else git clone -q https://github.com/Keyla-Cartagena/ADA-2026-2-GP01_Andres_y_Keyla.git /content/ADA-2026-2-GP01_Andres_y_Keyla; fi

import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO → Entorno de ejecución ▸ Reiniciar sesión, y continuar en la sección 2")

## 2 · Cargar y validar `entrega.json`

In [ ]:
import os, sys, json
from pathlib import Path
from collections import Counter

RAIZ = Path("/content/ADA-2026-2-GP01_Andres_y_Keyla")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from oraculo import Oraculo, CATALOGO, RANURAS, TEMPERATURAS

with open(RAIZ / "entrega.json", encoding="utf-8") as f:
    ENTREGA = json.load(f)
CONFIG = ENTREGA["config"]

# La configuración debe usar índices y temperatura que acepta la interfaz del oráculo.
assert set(CONFIG) == set(RANURAS) | {"temperatura"}, f"llaves inesperadas: {sorted(CONFIG)}"
for r in RANURAS:
    assert isinstance(CONFIG[r], int) and 0 <= CONFIG[r] < len(CATALOGO[r]), f"índice inválido en {r}"
assert CONFIG["temperatura"] in TEMPERATURAS, "temperatura fuera de las permitidas"

print("grupo:", ENTREGA["grupo"], "· semana:", ENTREGA["semana"])
print("configuración:", CONFIG)

## 3 · Modelo y datos de prueba

In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir

MODELO = "qwen17b"   # el mismo modelo con el que se hizo la búsqueda
modelo = cargar_modelo(MODELO)

candidatas = [os.environ.get("ORACULO_DATOS"),
              "/content/datos_prueba.json",
              "/content/drive/MyDrive/datos_prueba.json"]
RUTA_PRUEBA = next((Path(p) for p in candidatas if p and Path(p).exists()), None)

if RUTA_PRUEBA is not None:
    with open(RUTA_PRUEBA, encoding="utf-8") as f:
        prueba = json.load(f)
    # Si el archivo trae la marca de partición, no se evalúan las instancias descartadas.
    prueba = [x for x in prueba if x.get("split") != "descartada"]
    ORIGEN = str(RUTA_PRUEBA)
else:
    _, prueba = dividir(cargar_datos(str(RAIZ / "datos_visibles.json")))
    ORIGEN = "datos_visibles.json · partición de validación (no se encontró datos_prueba.json)"

print("datos de prueba:", ORIGEN)
print("instancias:", len(prueba))

## 4 · Evaluación

In [ ]:
oraculo = Oraculo(modelo, prueba, cache="/content/cache_evaluacion.json",
                  lote=16, presupuesto=32_000_000)
r = oraculo.evaluar(CONFIG, prueba, semilla=1)

aciertos = r.n - len(r.trazas)
print(f"precisión: {r.precision:.2%}  ({aciertos}/{r.n} instancias cumplen todas sus restricciones)")
print("\nprimera restricción incumplida, por familia:")
for familia, veces in Counter(t["violo"] for t in r.trazas).most_common(10):
    print(f"  {veces:3d}  {familia}")

In [ ]:
# Resultado en disco, por si se quiere descargar.
salida = {"grupo": ENTREGA["grupo"], "config": CONFIG, "modelo": MODELO, "datos": ORIGEN,
          "n": r.n, "aciertos": aciertos, "precision": r.precision,
          "fallos_por_familia": dict(Counter(t["violo"] for t in r.trazas))}
with open("/content/resultado_evaluacion.json", "w", encoding="utf-8") as f:
    json.dump(salida, f, ensure_ascii=False, indent=1)
salida